# 04 · NumPy 數值運算

用 NumPy 陣列（array）取代 Python 串列（list），學會以向量化（vectorization）思維做高效率數值運算。

## 學習目標
- 建立並理解 NumPy 陣列的形狀（shape）、資料型別（dtype），能變形（reshape）與拼接（concatenate）。
- 掌握廣播（broadcasting）規則，用形狀不同的陣列做整批運算。
- 運用布林遮罩（boolean mask）與進階索引（advanced indexing）做篩選與條件賦值。
- 使用統計與運算函式（mean、std、median、percentile、where、argsort 等）萃取資料特徵。
- 認識線性代數（linear algebra）與隨機亂數（random）模組，並能設定種子（seeding）重現結果。
- 體會向量化相對於迴圈（loop）的效能差異，培養避免明寫迴圈的習慣。

In [ ]:
# 中文字型設定（本書會畫圖才需要）
import os
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for fp in ['NotoSansCJK-Regular.otf', 'msjh.ttc', 'mingliu.ttc']:
    try:
        if os.path.exists(fp):
            fm.fontManager.addfont(fp)
    except Exception:
        pass
plt.rcParams['font.family'] = ['Microsoft JhengHei', 'Noto Sans CJK TC', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

## 陣列建立與形狀

NumPy 陣列（array）是一塊「同型別、有形狀」的連續資料，是後續一切數值運算的基礎。
相較於 Python 串列，陣列記住自己的形狀（shape）與資料型別（dtype），讓整批運算又快又省記憶體。

先建立「形狀與 dtype 決定運算行為」的直覺，並區分兩件常被搞混的事：
- 變形（reshape）：總元素數不變，只重新安排成不同維度。
- 拼接（concatenate）/ 垂直堆疊（vstack）：把多塊陣列接成更大的一塊。

形狀：`arr.reshape(列數, 欄數)`，其中 `列數 × 欄數` 必須等於原本的元素總數。

In [ ]:
# 概念：陣列建立 array creation、形狀 shape、變形 reshape、拼接 concatenate
import numpy as np

nums = np.arange(12)              # 造一串 0..11 的整數當示範用
print('原始一維陣列：', nums)
print('shape =', nums.shape, ' dtype =', nums.dtype)

grid = nums.reshape(3, 4)         # 變形成 3 列 4 欄，元素總數 12 不變
print('reshape 後：\n', grid)

# 造兩塊小陣列，準備示範堆疊與拼接
top = np.array([[1, 1, 1, 1]])    # 形狀 (1, 4)
bottom = np.array([[9, 9, 9, 9]])

# 注意：vstack 沿「列」方向往下疊，欄數要一致
stacked = np.vstack([top, grid, bottom])
print('vstack 疊出的矩陣：\n', stacked)

# 技巧：concatenate 用 axis 指定方向，axis=0 等同 vstack
joined = np.concatenate([top, bottom], axis=0)
print('concatenate(axis=0)：\n', joined)

## 廣播

廣播（broadcasting）讓形狀不同的陣列免寫迴圈即可逐元素運算，是 NumPy 高效率的核心機制。
運算時 NumPy 會把較小的陣列「自動延展」到對齊較大的形狀（形狀對齊 shape alignment）。

規則直覺：從最後一個維度往前比，對應維度若相等、或其中一個為 1，就能延展；否則失敗報錯。
純量（scalar）與陣列運算是最常見的廣播，例如「整批加上同一個數」。

In [ ]:
# 概念：廣播 broadcasting、形狀對齊 shape alignment、純量與陣列運算
import numpy as np

# 造一組假資料：4 列觀測、3 欄特徵
data = np.array([[10.0, 20.0, 30.0],
                 [12.0, 18.0, 33.0],
                 [11.0, 25.0, 27.0],
                 [13.0, 22.0, 30.0]])

col_mean = data.mean(axis=0)      # 沿列方向算每一欄的平均，得到形狀 (3,)
print('各欄平均：', col_mean)

# 注意：(4, 3) 與 (3,) 從最後維度對齊，3==3 可廣播；列向量自動套到每一列
centered = data - col_mean        # 每欄都減去自己的平均（去中心化）
print('去中心化後：\n', centered)

# 技巧：純量也是廣播，整批加 100 不需迴圈
print('整批加 100：\n', data + 100)

## 布林遮罩與進階索引

布林遮罩（boolean mask）是一個由 True / False 組成、形狀與資料相同的陣列。
用 `arr[mask]` 可一次取出所有 True 位置的元素，取代逐筆判斷的迴圈，是資料清理與篩選的關鍵手法。

同樣可做條件賦值（conditional assignment）：`arr[mask] = 新值`，把符合條件的元素整批改掉。

形狀：`mask = arr > 門檻` 產生遮罩；`arr[mask]` 取值；`arr[mask] = 0` 改值。

In [ ]:
# 概念：布林遮罩 boolean mask、條件篩選 arr[mask]、條件賦值 conditional assignment
import numpy as np

# 造一組假分數（含一個用來示範清理的負值）
scores = np.array([72, 58, 91, -5, 66, 45, 88])

pass_mask = scores >= 60          # 及格門檻 60，產生布林遮罩
print('遮罩：', pass_mask)
print('及格者：', scores[pass_mask])  # 一次取出所有 True 的元素

# 注意：條件賦值會就地修改原陣列，把不合理的負值統一改為 0
scores[scores < 0] = 0
print('清理負值後：', scores)

# 技巧：多條件要用 & | 並各自加括號（不能用 and / or）
band_mask = (scores >= 60) & (scores < 90)
print('60~89 分：', scores[band_mask])

## 統計摘要

統計摘要把一堆數字濃縮成少數幾個描述分布的特徵：平均（mean）、標準差（std）、中位數（median）、總和（sum）、百分位數（percentile）。

關鍵在於軸（axis）：對二維陣列，`axis=0` 沿列方向逐欄計算，`axis=1` 沿欄方向逐列計算，不指定則對全體計算。
平均易受極端值拉動，中位數較穩健，兩者並看能更了解資料分布。

In [ ]:
# 概念：平均 mean、標準差 std、中位數 median、百分位數 percentile、軸 axis
import numpy as np

# 造一個 3 列 4 欄的假數值矩陣
matrix = np.array([[ 5.0,  8.0,  6.0,  9.0],
                   [ 7.0, 50.0,  6.0,  8.0],
                   [ 6.0,  7.0,  5.0, 10.0]])

print('整體平均：', matrix.mean())
print('整體標準差：', round(matrix.std(), 3))

# 注意：axis=0 是「跨列、逐欄」，結果長度等於欄數 4
print('逐欄平均：', matrix.mean(axis=0))
print('逐欄中位數：', np.median(matrix, axis=0))

# 技巧：percentile 一次給多個分位點，回傳對應的陣列
print('第 25/50/75 百分位數：', np.percentile(matrix, [25, 50, 75]))

## 運算與轉換函式

這組函式讓常見的特徵處理都能向量化完成：
- 裁切（clip）：把值限制在上下界之內。
- 條件選擇（where）：依條件逐元素在兩個選項間挑一個。
- 排序索引（argsort）：回傳「排序後的索引」，常用來找名次。
- 最大值索引（argmax）：回傳最大值所在的位置。
- 次數統計（bincount）：統計非負整數標籤各自出現幾次。

In [ ]:
# 概念：裁切 clip、條件選擇 where、排序索引 argsort、最大值索引 argmax、次數統計 bincount
import numpy as np

values = np.array([3, -2, 18, 7, 25, 5, 9])

clipped = np.clip(values, 0, 20)  # 小於 0 變 0、大於 20 變 20
print('裁切到 [0, 20]：', clipped)

# 條件選擇：達標標 1、否則標 0
flags = np.where(values >= 10, 1, 0)
print('是否達 10：', flags)

# 注意：argsort 預設由小到大，取最大前三名要反轉索引
order_desc = np.argsort(values)[::-1]
print('最大前三名的索引：', order_desc[:3])
print('最大值所在位置 argmax：', np.argmax(values))

# 技巧：bincount 統計類別標籤次數（標籤須為非負整數）
labels = np.array([0, 1, 2, 1, 0, 1, 2])
print('各類別次數：', np.bincount(labels))

## 線性代數與隨機亂數

NumPy 內建線性代數（linear algebra）與隨機亂數（random）模組。
- 向量範數（linalg.norm）：衡量向量長度或殘差大小。
- 最小平方解（linalg.lstsq）：在過定方程中求最佳擬合直線。
- 亂數產生器：建議用新式 `default_rng`（舊式為 `RandomState`），並以種子（seeding）固定結果。

設定種子能讓每次執行得到相同的「隨機」資料，是實驗可重現（reproducibility）的關鍵。

In [ ]:
# 概念：亂數產生器 default_rng、種子 seeding、最小平方解 linalg.lstsq、向量範數 linalg.norm
import numpy as np

rng = np.random.default_rng(42)   # 固定種子 42，確保結果可重現

x = np.linspace(0, 10, 30)        # 0 到 10 取 30 個等距點
noise = rng.normal(0, 1.0, size=x.shape)  # 平均 0、標準差 1 的高斯雜訊
y = 2.0 * x + 1.0 + noise         # 真實關係 y=2x+1 再加雜訊

# 注意：lstsq 需要設計矩陣，第二欄補 1 代表截距項
A = np.column_stack([x, np.ones_like(x)])
(slope, intercept), *_ = np.linalg.lstsq(A, y, rcond=None)
print('擬合斜率：', round(slope, 3), ' 截距：', round(intercept, 3))

residual = y - (slope * x + intercept)
print('殘差範數：', round(np.linalg.norm(residual), 3))

plt.scatter(x, y, label='帶雜訊的點')
plt.plot(x, slope * x + intercept, color='red', label='lstsq 擬合直線')
plt.xlabel('x')
plt.ylabel('y')
plt.title('最小平方擬合')
plt.legend()
plt.show()

## 向量化與效能

向量化（vectorization）是把整批運算交給 NumPy 底層的 C 實作一次完成，免去 Python 層的逐筆迴圈（loop）。
資料量大時，向量化通常比明寫迴圈快上數十到數百倍，這是 NumPy 的主要價值。

工程習慣：動手寫迴圈前，先想「這能不能向量化」。下面用計時（timing）比較同一計算的兩種寫法。

In [ ]:
# 概念：向量化 vectorization、迴圈 loop、計時 timing、效能差異 performance
import numpy as np
import time

big = np.arange(1_000_000, dtype=np.float64)  # 造一個百萬元素的大陣列

# 寫法一：Python 迴圈逐筆累加平方
t0 = time.perf_counter()
loop_sum = 0.0
for v in big:
    loop_sum += v * v
loop_time = time.perf_counter() - t0

# 寫法二：向量化，一行算完所有平方再加總
t0 = time.perf_counter()
vec_sum = np.sum(big * big)
vec_time = time.perf_counter() - t0

print('迴圈結果：', loop_sum, ' 耗時：', round(loop_time, 4), '秒')
print('向量化結果：', vec_sum, ' 耗時：', round(vec_time, 4), '秒')
# 技巧：結果相同但向量化快很多，倍率視機器而定
print('向量化約快', round(loop_time / max(vec_time, 1e-9), 1), '倍')

## 練習

以下三題由淺入深，皆為複合型但簡短可快速完成。資料一律在題目內用 numpy 自造，不引用任何外部檔案。

In [ ]:
# TODO 1 ·（簡單）計算建築樓地板面積與容積率（整合：陣列建立 array creation + 廣播 broadcasting）
#   情境：手上有一小批建築的占地面積 footprint area 與樓層數 floors，
#         想快速算出每棟的樓地板面積 GFA 與容積率 FAR。
#   要求：
#     1. 用 numpy 自造兩個形狀相同的一維陣列：footprint（占地面積，平方公尺，5 棟）與 floors（樓層數）。
#     2. 用逐元素相乘（廣播）算出樓地板面積 GFA = footprint * floors。
#     3. 自造一維陣列 lot（各棟基地面積），用 GFA / lot 算出容積率 FAR。
#   驗收：應看到一個與輸入等長的 FAR 陣列，每個值都是合理正數（例如 1.0 到 5.0 之間）。
# TODO: 在這裡完成

In [ ]:
# TODO 2 ·（中間）街廓網格的建築高度聚合
#   （整合：陣列建立 array creation + 廣播 broadcasting + 布林遮罩 boolean mask + 統計摘要 statistical summary）
#   情境：把一個街廓 block 切成數個網格 cell，需統整每格內建築樓高 height 的分布，並標記過高的建築。
#   要求：
#     1. 用 numpy 自造一個二維陣列 heights（形狀 4 列 x 6 欄，代表 4 格各 6 棟建築的樓高，公尺）。
#     2. 沿軸 axis 算出每一格（每列）的平均 mean、標準差 std 與第 90 百分位數 percentile。
#     3. 用廣播把每棟樓高減去「該格平均」，得到每棟相對於所在網格的高度差。
#     4. 用布林遮罩找出全體中超過「整體平均 + 一個標準差」的過高建築，並回報其數量。
#   驗收：應看到每格的三項統計值，以及一個過高建築的布林遮罩與其總數。
# TODO: 在這裡完成

In [ ]:
# TODO 3 ·（稍難）政策高度乘數下的容積達標篩選
#   （整合：陣列建立 array creation + 廣播 broadcasting + 條件選擇 where + 統計摘要 statistical summary + 排序索引 argsort）
#   情境：某政策對不同用途分類 land-use 給予不同高度乘數 height multiplier，
#         需評估調整後哪些建築達到目標容積率 FAR，並排出潛力名單。
#   要求：
#     1. 用 numpy 自造：floors（樓層數）、footprint（占地面積）、lot（基地面積），
#        以及整數陣列 usecode（用途標籤，0=住宅、1=商業、2=工業）。
#     2. 建長度 3 的乘數陣列 multiplier，用進階索引 multiplier[usecode] 把每棟對應到自己的乘數，
#        再用 where 對特定用途（如商業）再加成，得到調整後的有效樓層數。
#     3. 用廣播算出調整後 GFA 與 FAR，並用布林遮罩篩出 FAR 達門檻（如 >= 3.0）的建築。
#     4. 對達標建築依 FAR 由高到低用 argsort 排序，輸出前 3 名的索引作為開發潛力名單。
#   驗收：應看到達標建築的布林遮罩、達標數量，以及依 FAR 排序後的前 3 名建築索引。
# TODO: 在這裡完成

## 小結

- NumPy 陣列以形狀（shape）與資料型別（dtype）為核心，reshape 改排列、concatenate / vstack 接資料。
- 廣播（broadcasting）讓不同形狀的陣列免迴圈逐元素運算，是高效率的基礎機制。
- 布林遮罩（boolean mask）與進階索引（advanced indexing）用條件直接取值與改值，取代逐筆判斷。
- 統計函式搭配軸（axis）能逐欄 / 逐列萃取平均、標準差、中位數與百分位數等特徵。
- clip、where、argsort、argmax、bincount 是常用的向量化特徵處理工具。
- 線性代數與亂數模組搭配種子（seeding）可做可重現的擬合與實驗。
- 動手寫迴圈前先想能否向量化，是寫出高效數值程式的工程習慣。